In [2]:
import os
import sys
import inspect

currentdir = os.path.dirname(os.path.abspath(inspect.getfile(inspect.currentframe())))
parentdir = os.path.dirname(currentdir)
sys.path.insert(0, parentdir) 

import pickle
import numpy as np # type: ignore
import matplotlib.pyplot as plt
import pandas as pd
import string
from sim import simulate_mechanism
import random
import dask.dataframe as dd
from dask import delayed
import glob

In [3]:
#opens all the mechanisms to use in the simulation
mechanism_file = open("dataset","rb")
mechs = pickle.load(mechanism_file)
mechanism_file.close()

In [4]:
def get_linkage(A, node_types):
    n_nodes = len(node_types)
    fixed_nodes = np.where(node_types)[0]

    new_matrix = np.zeros((len(A),len(A)))

    # Iterate over the elements of matrix A
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            # Check the conditions and update the new matrix
            if i > j and A[i, j]:
                new_matrix[i, j] = A[i, j]
    
    positions = []

    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            # Check if the value is 1
            if A[i, j] == 1:
                # Append the position (i, j) to the list
                positions.append([i, j])

    connectivity = []
    sums_encountered = set()

    for pair in positions:
        pair_sum = sum(pair)
        if pair_sum not in sums_encountered:
            connectivity.append(pair)
            sums_encountered.add(pair_sum)
    area = 2e-6 #mm^2
    st_steel = 1.8e10 #GPa

    structure = []
    for i in range (n_nodes):
        #print(connectivity[i][0], connectivity[i][1], area, st_steel)
        structure.append([connectivity[i][0], connectivity[i][1], area, st_steel])
    columns = ['start', 'end', 'area', 'material']
    struct_df = pd.DataFrame(structure, columns=columns)
    struct_df = struct_df.iloc[1:]
    
    fixed_x = []
    fixed_y = []

    for i in range (n_nodes):
        if i in fixed_nodes or i == 1:
            fixed_x.append(0)
            fixed_y.append(0)
        else:
            fixed_x.append(np.nan)
            fixed_y.append(np.nan)

    force = 10000 #100 N
    force_dir = random.choice([True, False])
    force_node = random.choice([i for i in range(n_nodes) if i not in fixed_nodes])

    force_x = []
    force_y = []

    for i in range (n_nodes):
        if i == force_node:
            if force_dir:
                force_x.append(force)
                force_y.append(0)
            else:
                force_x.append(0)
                force_y.append(force)
        else:
            force_x.append(0)
            force_y.append(0)

    nodes_df = pd.DataFrame({'x': None, 'y': None, 'disp_x': fixed_x,'disp_y':fixed_y , 'force_x': force_x, 'force_y': force_y})

    return nodes_df, struct_df

In [5]:
def compute(struct):
    start = struct['start']
    end = struct['end']

    x_coordinate_start = nodes.loc[start, 'x']
    y_coordinate_start = nodes.loc[start, 'y']

    x_coordinate_end = nodes.loc[end, 'x']
    y_coordinate_end = nodes.loc[end, 'y']

    delta_x = x_coordinate_end - x_coordinate_start
    delta_y = y_coordinate_end - y_coordinate_start
    length = np.sqrt(delta_x**2 + delta_y**2)

    stiffness = struct['material'] * struct['area'] / length
    c = delta_x / length
    s = delta_y / length

    R = np.array([
        [c*c, c*s, -c*c, -c*s],
        [c*s, s*s, -c*s, -s*s],
        [-c*c, -c*s, c*c, c*s],
        [-c*s, -s*s, c*s, s*s]])

    return length, stiffness, R

In [6]:
def compute_global(struct):
    N = len(nodes)
    indicies = np.arange(2*N)
    indicies = indicies.reshape((-1, 2))
    K = np.zeros((2*N, 2*N))
    start = struct['start']
    end = struct['end']
    indicies = np.hstack([indicies[start], indicies[end]])
    K[np.ix_(indicies, indicies)] = struct['stiffness'] * struct['R']

    return K

In [7]:
def partitionK(K, A, B):
    KAA = K[np.ix_(A, A)]
    KAB = K[np.ix_(A, B)]
    KBA = K[np.ix_(B, A)]
    KBB = K[np.ix_(B, B)]
    return KAA, KAB, KBA, KBB

In [8]:
def find_steps(A, x0, node_types):
    M = (A, x0, node_types)
    n_steps = 900

    output = simulate_mechanism(M, n = n_steps) 

    all_positions = []

    for i in range(n_steps):
        for j in range(len(output[0])):
            position = output[0][j][i]
            all_positions.append(position)
        
    all_positions = np.array(all_positions)

In [9]:
total_links = 100000 # Set to the total number of mechanisms you want to simulate
fail_start = 0
i = 0

def save_batch_data(X_batch, Y_batch, batch_num):
    """Save a batch of data incrementally using delayed operations"""
    X_df = pd.DataFrame(X_batch)
    Y_df = pd.DataFrame(Y_batch)
    
    # Save as parquet files for efficiency
    X_df.to_parquet(f"X_batch_{batch_num:06d}.parquet")
    Y_df.to_parquet(f"Y_batch_{batch_num:06d}.parquet")
    
    return len(X_df)

@delayed
def process_mechanism(mech_data, mech_index):
    """Process a single mechanism using Dask delayed"""
    try:
        A, x0, node_types = mech_data
        M = (A, x0, node_types)
        n_steps = 30

        output = simulate_mechanism(M, n=n_steps)
        nodes, struct = get_linkage(A, node_types)

        # Process all steps for this mechanism
        for j in range(n_steps):
            step_positions = []
            for k in range(len(output[0])):
                position = output[0][k][j]
                step_positions.append(position)

            step_positions = np.array(step_positions)
            
            nodes["x"] = step_positions[:, 0]
            nodes["y"] = step_positions[:, 1]

            nodes = pd.DataFrame(nodes, columns=['x', 'y', 'disp_x', 'disp_y', 'force_x', 'force_y'])
            struct = pd.DataFrame(struct, columns=['start', 'end', 'area', 'material'])
            
            struct[['length', 'stiffness', 'R']] = struct.apply(compute, axis=1, result_type='expand')
            K = struct.apply(compute_global, axis=1).sum()
            K.round(1)

            U = nodes[['disp_x', 'disp_y']].to_numpy()
            U = U.ravel()
            A_indices = np.isnan(U)
            
            P = nodes[['force_x', 'force_y']].to_numpy()
            P = P.ravel()
            B_indices = np.isnan(P)

            KAA, KAB, KBA, KBB = partitionK(K, A_indices, B_indices)

            UB = U[B_indices]
            PA = P[A_indices]

            UA = np.dot(np.linalg.pinv(KAA), (PA - np.dot(KAB, UB)))
            U[A_indices] = UA

            PB = np.dot(KBA, UA) + np.dot(KBB, UB)
            P[B_indices] = PB

            result = nodes
            result[['disp_x', 'disp_y']] = U.reshape(-1, 2)

        # Find the node that had force on it and the direction of the force
        force_node = nodes[(nodes['force_x'] != 0) | (nodes['force_y'] != 0)].index[0]
        force_direction = 0 if nodes.loc[force_node, 'force_x'] != 0 else 1

        # Prepare the data
        X = {'x': nodes['x'].tolist(), 'y': nodes['y'].tolist(), 'num_nodes': len(nodes), 'force_node': force_node, 'force_direction': force_direction}
        Y = {'avg_xdisp': nodes['disp_x'].tolist(), 'avg_ydisp': nodes['disp_y'].tolist()}

        return X, Y, mech_index

    except Exception as e:
        print(f"Failed on mechanism {mech_index}: {e}")
        return None, None, mech_index

def main():
    # Configuration
    batch_size = 100  # Process and save data in batches
    
    # global variables
    global total_links
    global i
    global nodes
    global struct
    global fail_start
    
    # Initialize Dask client for better performance monitoring
    try:
        from dask.distributed import Client
        client = Client(n_workers=2, threads_per_worker=2, memory_limit='2GB')
        print(f"Dask client: {client}")
    except:
        print("Running without Dask distributed client")
    
    batch_num = 0
    processed_count = 0
    
    # Process data in batches
    for batch_start in range(fail_start, total_links, batch_size):
        batch_end = min(batch_start + batch_size, total_links)
        print(f"Processing batch {batch_num}: mechanisms {batch_start} to {batch_end-1}")
        
        # Create delayed tasks for this batch
        delayed_tasks = []
        for i in range(batch_start, batch_end):
            if i < len(mechs):
                delayed_tasks.append(process_mechanism(mechs[i], i))
        
        # Compute the batch
        results = dd.compute(*delayed_tasks)
        
        # Separate successful results
        X_batch = []
        Y_batch = []
        
        for X, Y, mech_idx in results:
            if X is not None and Y is not None:
                X_batch.append(X)
                Y_batch.append(Y)
                processed_count += 1
        
        # Save the batch if we have data
        if X_batch:
            save_batch_data(X_batch, Y_batch, batch_num)
            print(f"Saved batch {batch_num} with {len(X_batch)} records")
        
        batch_num += 1
    
    print(f"Total mechanisms processed successfully: {processed_count}")
    
    # Combine all batch files
    print("Combining all batch files...")
    combine_batch_files()

def combine_batch_files():
    """Combine all batch files into final datasets using Dask"""
    # Find all batch files
    x_files = glob.glob("X_batch_*.parquet")
    y_files = glob.glob("Y_batch_*.parquet")
    
    if not x_files or not y_files:
        print("No batch files found to combine")
        return
    
    # Sort files to ensure correct order
    x_files.sort()
    y_files.sort()
    
    print(f"Found {len(x_files)} X batch files and {len(y_files)} Y batch files")
    
    # Use Dask to read and combine files efficiently
    try:
        X_result = dd.read_parquet("X_batch_*.parquet")
        Y_result = dd.read_parquet("Y_batch_*.parquet")
        
        # Save final results as single files
        X_result.to_csv("2X_results.csv", single_file=True)
        Y_result.to_csv("2Y_results.csv", single_file=True)
        
        # Create final combined dataset
        X_pd = X_result.compute()
        Y_pd = Y_result.compute()
        
        final = pd.concat([X_pd.reset_index(drop=True), Y_pd.reset_index(drop=True)], axis=1)
        final['avg_disp'] = final.apply(lambda row: row['avg_xdisp'] + row['avg_ydisp'], axis=1)
        final.drop(columns=['avg_xdisp', 'avg_ydisp'], inplace=True)
        
        display(final)
        print(f"Final dataset shape: {final.shape}")
        if len(Y_pd) > 0:
            print(f"Length of first Y record: {len(Y_pd.iloc[0, 0])}")
        
        final.to_csv("2final.csv", index=False)
        print("Final combined dataset saved to 2final.csv")
        
        # Clean up batch files
        cleanup_batch_files()
        print("Batch files cleaned up")
        
    except Exception as e:
        print(f"Error combining files: {e}")
        # Fallback: combine using pandas
        combine_batch_files_pandas()

def combine_batch_files_pandas():
    """Fallback method to combine batch files using pandas"""
    x_files = glob.glob("X_batch_*.parquet")
    y_files = glob.glob("Y_batch_*.parquet")
    
    x_dfs = [pd.read_parquet(f) for f in sorted(x_files)]
    y_dfs = [pd.read_parquet(f) for f in sorted(y_files)]
    
    X_result = pd.concat(x_dfs, ignore_index=True)
    Y_result = pd.concat(y_dfs, ignore_index=True)
    
    X_result.to_csv("2X_results.csv", index=False)
    Y_result.to_csv("2Y_results.csv", index=False)
    
    final = pd.concat([X_result, Y_result], axis=1)
    final['avg_disp'] = final.apply(lambda row: row['avg_xdisp'] + row['avg_ydisp'], axis=1)
    final.drop(columns=['avg_xdisp', 'avg_ydisp'], inplace=True)
    
    display(final)
    final.to_csv("2final.csv", index=False)

def cleanup_batch_files():
    """Remove temporary batch files"""
    batch_files = glob.glob("*_batch_*.parquet")
    for file in batch_files:
        try:
            os.remove(file)
            print(f"Removed {file}")
        except Exception as e:
            print(f"Could not remove {file}: {e}")

# Function to monitor memory usage (optional)
def get_memory_usage():
    """Get current memory usage"""
    import psutil
    process = psutil.Process(os.getpid())
    memory_info = process.memory_info()
    return memory_info.rss / 1024 / 1024  # Convert to MB

In [10]:
main()

Dask client: <Client: 'tcp://127.0.0.1:53648' processes=2 threads=4, memory=3.73 GiB>
Processing batch 0: mechanisms 0 to 99
Processing batch 1: mechanisms 100 to 199
Processing batch 1: mechanisms 100 to 199
Processing batch 2: mechanisms 200 to 299
Processing batch 2: mechanisms 200 to 299
Processing batch 3: mechanisms 300 to 399
Processing batch 3: mechanisms 300 to 399
Processing batch 4: mechanisms 400 to 499
Processing batch 4: mechanisms 400 to 499
Processing batch 5: mechanisms 500 to 599
Processing batch 5: mechanisms 500 to 599
Processing batch 6: mechanisms 600 to 699
Processing batch 6: mechanisms 600 to 699
Processing batch 7: mechanisms 700 to 799
Processing batch 7: mechanisms 700 to 799
Processing batch 8: mechanisms 800 to 899
Processing batch 8: mechanisms 800 to 899
Processing batch 9: mechanisms 900 to 999
Processing batch 9: mechanisms 900 to 999
Processing batch 10: mechanisms 1000 to 1099
Processing batch 10: mechanisms 1000 to 1099
Processing batch 11: mechanis

In [11]:
# def visualize_motion_and_displacements(mechs, total_links, scale_factor=10):
#     for idx in range(total_links):
#         try:
#             # Extract the mechanism data
#             A, x0, node_types = mechs[idx]
#             nodes, struct = get_linkage(A, node_types)
#             n_steps = 900
#             M = (A, x0, node_types)
#             output = simulate_mechanism(M, n=n_steps)

#             # Initialize the plot
#             fig, ax = plt.subplots(figsize=(8, 8))
#             ax.set_aspect('equal')
#             plt.xlabel('X')
#             plt.ylabel('Y')
#             plt.title(f'Linkage Motion and Displacements - Mechanism {idx + 1}')

#             # Plot the motion over time with more fading and blue tint
#             for step in range(n_steps):
#                 step_positions = []
#                 for k in range(len(output[0])):
#                     position = output[0][k][step]
#                     step_positions.append(position)

#                 step_positions = np.array(step_positions)
#                 nodes["x"] = step_positions[:, 0]
#                 nodes["y"] = step_positions[:, 1]

#                 # Adjust fade factor to make fading more pronounced
#                 fade_factor = (1 - step / n_steps) * 0.125  # Maximum brightness is 0.3
#                 faded_color = (fade_factor, fade_factor, 1.0, fade_factor)  # Blue tint with fading
#                 for _, row in struct.iterrows():
#                     start_node = nodes.loc[row['start']]
#                     end_node = nodes.loc[row['end']]
#                     ax.plot([start_node['x'], end_node['x']], [start_node['y'], end_node['y']], color=faded_color)

#             # Overlay the actual linkage in blue
#             for _, row in struct.iterrows():
#                 start_node = nodes.loc[row['start']]
#                 end_node = nodes.loc[row['end']]
#                 ax.plot([start_node['x'], end_node['x']], [start_node['y'], end_node['y']], 'bo-')

#             # Overlay the exaggerated displacements in red
#             for _, row in nodes.iterrows():
#                 # Arrow from original position to displaced position
#                 ax.arrow(
#                     row['x'], row['y'], 
#                     row['disp_x'] * scale_factor, row['disp_y'] * scale_factor, 
#                     color='red', head_width=0.02, head_length=0.03
#                 )

#                 # Mark the displaced position
#                 displaced_x = row['x'] + row['disp_x'] * scale_factor
#                 displaced_y = row['y'] + row['disp_y'] * scale_factor
#                 ax.plot(displaced_x, displaced_y, 'go')  # Green dot for displaced position

#             # Display the plot
#             plt.show()

#         except Exception as e:
#             print(f"Failed to visualize mechanism {idx + 1}: {e}")

# # Call the function to generate the visualizations with exaggerated displacements
# visualize_motion_and_displacements(mechs, total_links, scale_factor=1000)

In [12]:
# import matplotlib.pyplot as plt

# def plot_linkage(nodes, struct):
#     fig, ax = plt.subplots()
#     for _, row in struct.iterrows():
#         start_node = nodes.loc[row['start']]
#         end_node = nodes.loc[row['end']]
#         # Plot the nodes
#         ax.plot(nodes['x'], nodes['y'], 'ro')
#         # Plot the linkage between start and end nodes
#         ax.plot([start_node['x'], end_node['x']], [start_node['y'], end_node['y']], 'bo-')
    
#     ax.set_aspect('equal')
#     plt.xlabel('X')
#     plt.ylabel('Y')
#     plt.title('Linkage and Displacements')
#     plt.show()

# plot_linkage(nodes, struct)